In [1]:
import pandas as pd
import numpy as np
import re, unicodedata, json
from itertools import combinations
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_white"
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
colorscale="haline"


# **Bloque A — Varios HTML (uno por plot) con dropdown por figura, ya “bonito”**

In [2]:
# =========================
# CONFIG
# =========================
SCOPUS_CSV_PATH = "scopus_export_Apr 15-2026_4b53d047-aa5f-439b-8607-426be5a16419.csv"
FACULTY_XLSX_PATH = "Base de Datos Scopus 2024.xlsx"
FACULTY_SHEET = "Hoja1"

START_YEAR = 2022
OUT_ROOT = Path("utb_scopus_dashboard_single_pretty/")
OUT_DIR = OUT_ROOT

TOP_SCHOOLS = 18
TOP_AUTHORS = 20
TOP_PAIRS = 30
DOC_TYPES_ORDER = ["Article", "Conference", "Review", "Other"]

In [3]:
Actu=SCOPUS_CSV_PATH.split("_")[2]

In [4]:
# =========================
# Helpers
# =========================
def _norm(s) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _split_semicol(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return []
    xs = [x.strip() for x in str(s).split(";")]
    return [x for x in xs if x != ""]

def _parse_author_aff_entry(entry: str):
    if entry is None or (isinstance(entry, float) and np.isnan(entry)):
        return (None, None)
    entry = str(entry).strip()
    if entry == "" or _norm(entry) in ("nan", "null", "0"):
        return (None, None)
    parts = [p.strip() for p in entry.split(",", 2)]
    if len(parts) >= 3:
        last, first, aff = parts[0], parts[1], parts[2]
        name = f"{last}, {first}".strip(", ")
    elif len(parts) == 2:
        last, first = parts
        name = f"{last}, {first}".strip(", ")
        aff = None
    else:
        name = parts[0]
        aff = None
    if _norm(name) in ("", "nan", "null", "0"):
        name = None
    if isinstance(aff, str) and _norm(aff) in ("", "nan", "null", "0"):
        aff = None
    return name, aff

def _extract_authorid_from_url(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    m = re.search(r"authorId=(\d+)", str(x))
    return m.group(1) if m else None

def _canon_author_id(v):
    if pd.isna(v):
        return None
    try:
        iv = int(v)
        return None if iv == 0 else str(iv)
    except Exception:
        s = str(v).strip()
        return None if s in ("", "0", "nan") else s

def _doc_type_bucket(dt) -> str:
    t = _norm(dt)
    if t == "":
        return "Other"
    if "review" in t:
        return "Review"
    if "conference" in t or "proceeding" in t:
        return "Conference"
    if "article" in t:
        return "Article"
    return "Other"

# ── Title-case helper ──────────────────────────────────────────────────────────
_LOWER_WORDS_ES = {"de", "del", "la", "el", "las", "los", "y", "e", "a",
                   "con", "en", "o", "por", "para", "al"}

def _title_case(s) -> str:
    """Convert ALL-CAPS string to Title Case, keeping Spanish prepositions lower."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return s
    words = str(s).strip().split()
    result = []
    for i, w in enumerate(words):
        clean = w.rstrip(",.;:")
        suffix = w[len(clean):]
        wl = clean.lower()
        if i > 0 and wl in _LOWER_WORDS_ES:
            result.append(wl + suffix)
        else:
            result.append(wl.capitalize() + suffix)
    return " ".join(result)


In [5]:
# =========================
# Load faculty (planta UTB)
# =========================
faculty_raw = pd.read_excel(FACULTY_XLSX_PATH, sheet_name=FACULTY_SHEET).copy()
faculty_raw["author_id_from_url"] = faculty_raw.get("SCOPUS", pd.Series([None]*len(faculty_raw))).apply(_extract_authorid_from_url)
faculty_raw["author_id"] = faculty_raw.get("ID SCOPUS", pd.Series([None]*len(faculty_raw))).apply(_canon_author_id)
faculty_raw.loc[faculty_raw["author_id"].isna(), "author_id"] = faculty_raw.loc[faculty_raw["author_id"].isna(), "author_id_from_url"]

faculty_valid = faculty_raw[faculty_raw["author_id"].notna()].copy()
faculty_valid["author_id"] = faculty_valid["author_id"].astype(str).str.strip()
faculty_valid = faculty_valid.drop_duplicates(subset=["author_id"]).reset_index(drop=True)

faculty_missing_id = faculty_raw[faculty_raw["author_id"].isna()].copy().reset_index(drop=True)

for col in ["DOCENTE", "ESCUELA"]:
    if col not in faculty_valid.columns:
        faculty_valid[col] = None

faculty_ids = set(faculty_valid["author_id"].tolist())
name_map = {k: _title_case(v) for k, v in faculty_valid.set_index("author_id")["DOCENTE"].to_dict().items()}
school_map = {k: _title_case(v) for k, v in faculty_valid.set_index("author_id")["ESCUELA"].to_dict().items()}

# =========================
# Load scopus + filter
# =========================
df = pd.read_csv(SCOPUS_CSV_PATH, encoding="utf-8-sig").copy()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df[df["Year"].notna()].copy()
df["Year"] = df["Year"].astype(int)
df = df[df["Year"] >= START_YEAR].copy()

if "Document Type" not in df.columns:
    df["Document Type"] = None
df["doc_type3"] = df["Document Type"].apply(_doc_type_bucket)

years = sorted(df["Year"].unique().tolist())
year_sels = ["ALL"] + [str(y) for y in years]

# ── Title-case helper ──────────────────────────────────────────────────────────
_LOWER_WORDS_ES = {"de", "del", "la", "el", "las", "los", "y", "e", "a",
                   "con", "en", "o", "por", "para", "al"}

def _title_case(s) -> str:
    """Convert ALL-CAPS string to Title Case, keeping Spanish prepositions lower."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return s
    words = str(s).strip().split()
    result = []
    for i, w in enumerate(words):
        # strip trailing punctuation before deciding
        clean = w.rstrip(",.;:")
        suffix = w[len(clean):]
        wl = clean.lower()
        if i > 0 and wl in _LOWER_WORDS_ES:
            result.append(wl + suffix)
        else:
            result.append(wl.capitalize() + suffix)
    return " ".join(result)


In [6]:
# =========================
# Build authors_long (from filtered papers)
# =========================
records = []
for _, r in df.iterrows():
    eid = r.get("EID")
    year = int(r.get("Year"))
    dt3 = r.get("doc_type3")

    ids = _split_semicol(r.get("Author(s) ID"))
    awas = _split_semicol(r.get("Authors with affiliations"))
    short = _split_semicol(r.get("Authors"))

    n = max(len(ids), len(awas), len(short))
    ids += [None] * (n - len(ids))
    awas += [None] * (n - len(awas))
    short += [None] * (n - len(short))

    for i in range(n):
        name, _aff = _parse_author_aff_entry(awas[i]) if awas[i] is not None else (None, None)
        if name is None and short[i] is not None and str(short[i]).strip() != "":
            name = str(short[i]).strip()

        author_id = str(ids[i]).strip() if ids[i] is not None else None
        if author_id in ("", "nan", "None"):
            author_id = None

        records.append(
            {
                "EID": eid,
                "Year": year,
                "doc_type3": dt3,
                "author_position": i + 1,
                "author_id": author_id,
                "author_name": name,
            }
        )

authors_long = pd.DataFrame.from_records(records)
authors_long = authors_long.dropna(subset=["author_id", "author_name"], how="all").reset_index(drop=True)
authors_long["author_id"] = authors_long["author_id"].astype(str).str.strip()

# Only faculty (planta)
utb_planta_long = authors_long[authors_long["author_id"].isin(faculty_ids)].copy()
utb_planta_long["DOCENTE"] = utb_planta_long["author_id"].map(name_map).fillna(utb_planta_long["author_id"])
utb_planta_long["ESCUELA"] = utb_planta_long["author_id"].map(school_map).fillna("ESCUELA (missing)")

# Unique-paper “credit” datasets
planta_school_papers = utb_planta_long[["EID", "Year", "ESCUELA", "doc_type3"]].drop_duplicates()
planta_author_papers = utb_planta_long[["EID", "Year", "author_id", "DOCENTE", "ESCUELA", "doc_type3"]].drop_duplicates()

# =========================
# Aggregators
# =========================
def _filter_year(df_in: pd.DataFrame, year_sel: str) -> pd.DataFrame:
    if year_sel == "ALL":
        return df_in
    return df_in[df_in["Year"] == int(year_sel)]

def school_doc_counts(year_sel: str):
    base = _filter_year(planta_school_papers, year_sel).copy()
    counts = base.groupby(["ESCUELA", "doc_type3"]).size().reset_index(name="n_docs")

    tot = counts.groupby("ESCUELA")["n_docs"].sum().reset_index(name="total")
    top = tot.sort_values("total", ascending=False).head(TOP_SCHOOLS)["ESCUELA"].tolist()

    counts["ESCUELA2"] = np.where(counts["ESCUELA"].isin(top), counts["ESCUELA"], "Other")
    counts2 = (
        counts.groupby(["ESCUELA2", "doc_type3"], as_index=False)["n_docs"]
        .sum()
        .rename(columns={"ESCUELA2": "ESCUELA"})
    )
    counts2["doc_type3"] = pd.Categorical(counts2["doc_type3"], categories=DOC_TYPES_ORDER, ordered=True)

    # orden por total DESC (para que el más grande quede abajo en barra horizontal)
    order = counts2.groupby("ESCUELA")["n_docs"].sum().sort_values(ascending=False).index.tolist()
    return counts2, order

def author_doc_counts(year_sel: str):
    base = _filter_year(planta_author_papers, year_sel).copy()

    # papers únicos por (autor, tipo)
    counts = (
        base.groupby(["author_id", "DOCENTE", "ESCUELA", "doc_type3"])
        .size()
        .reset_index(name="n_docs")
    )

    # total por autor (en el año seleccionado)
    tot = (
        counts.groupby(["author_id", "DOCENTE", "ESCUELA"], as_index=False)["n_docs"]
        .sum()
        .rename(columns={"n_docs": "total"})
    )

    # Top N autores (en el año seleccionado)
    top_ids = tot.sort_values("total", ascending=False).head(TOP_AUTHORS)["author_id"].tolist()
    counts = counts[counts["author_id"].isin(top_ids)].copy()

    counts["doc_type3"] = pd.Categorical(counts["doc_type3"], categories=DOC_TYPES_ORDER, ordered=True)

    # ORDEN (fijo y correcto): suma por DOCENTE y ordena por total del año
    tot2 = counts.groupby("DOCENTE", as_index=False)["n_docs"].sum()
    order = tot2.sort_values("n_docs", ascending=True)["DOCENTE"].tolist()  # ascending=True: mayor queda abajo

    return counts, order

def heatmap_school_doctype(year_sel: str):
    counts, school_order = school_doc_counts(year_sel)

    piv = counts.pivot_table(index="ESCUELA", columns="doc_type3", values="n_docs", fill_value=0)

    # Reindex de filas y columnas + fill_value (evita NaNs desde la fuente)
    piv = piv.reindex(index=school_order, columns=DOC_TYPES_ORDER, fill_value=0)

    # Forzar a numérico por si algo quedó como object
    piv = piv.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    # add totals row/col
    piv["TOTAL"] = piv.sum(axis=1)
    piv.loc["TOTAL"] = piv.sum(axis=0)

    # debug opcional
    # print(year_sel, "NaNs:", int(piv.isna().sum().sum()))

    return piv

# =========================
# Pairs faculty–faculty
# =========================
paper_fac = utb_planta_long[["EID", "Year", "author_id"]].drop_duplicates()

pairs_rows = []
for eid, g in paper_fac.groupby("EID"):
    y = int(g["Year"].iloc[0])
    ids = sorted(set(g["author_id"].astype(str).tolist()))
    if len(ids) < 2:
        continue
    for a, b in combinations(ids, 2):
        pairs_rows.append((y, a, b, eid))

pairs_df = pd.DataFrame(pairs_rows, columns=["Year", "author_id_a", "author_id_b", "EID"])
pairs_counts_year = (
    pairs_df.groupby(["Year", "author_id_a", "author_id_b"])
    .agg(n_shared=("EID", "nunique"))
    .reset_index()
)
pairs_counts_all = (
    pairs_df.groupby(["author_id_a", "author_id_b"])
    .agg(n_shared=("EID", "nunique"))
    .reset_index()
)

def top_pairs(year_sel: str) -> pd.DataFrame:
    if year_sel == "ALL":
        c = pairs_counts_all.copy()
    else:
        y = int(year_sel)
        c = pairs_counts_year[pairs_counts_year["Year"] == y].drop(columns=["Year"]).copy()

    if c.empty:
        return c

    c["DOCENTE_A"] = c["author_id_a"].map(name_map).fillna(c["author_id_a"])
    c["DOCENTE_B"] = c["author_id_b"].map(name_map).fillna(c["author_id_b"])
    c["ESCUELA_A"] = c["author_id_a"].map(school_map)
    c["ESCUELA_B"] = c["author_id_b"].map(school_map)
    c["pair"] = c["DOCENTE_A"] + " — " + c["DOCENTE_B"]

    return c.sort_values("n_shared", ascending=False).head(TOP_PAIRS).copy()


In [7]:
# =========================
# Plotly – Enhanced visualizations
# =========================
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Color palette ─────────────────────────────────────────────────────────────
DOC_TYPE_COLORS = {
    "Article":    "#2563EB",
    "Conference": "#EA580C",
    "Review":     "#059669",
    "Other":      "#9CA3AF",
}
_FONT = "'Segoe UI', 'Inter', Arial, sans-serif"

# ── Layout helpers ─────────────────────────────────────────────────────────────
def _base_layout(title, width=1200, height=700, lm=270, a=0):
    return dict(
        title=dict(text="<b>"+title+"</b>",
                   font=dict(size=21, color="#0F172A", family=_FONT),
                   x=0.5, xanchor="center", pad=dict(b=6)),
        width=width, height=height,
        font=dict(size=14, family=_FONT, color="#334155"),
        margin=dict(l=lm, r=60, t=120, b=90),
        plot_bgcolor="#F8FAFC", paper_bgcolor="white",
        legend=dict(title=dict(text="<b>Tipo de documento</b>",
                               font=dict(size=13, color="#475569")),
                    orientation="h", x=0.0, xanchor="left", y=-0.16-a, yanchor="top",
                    bgcolor="rgba(255,255,255,0.9)",
                    bordercolor="rgba(203,213,225,0.8)", borderwidth=1),
        xaxis=dict(showgrid=True, gridcolor="rgba(203,213,225,0.5)", gridwidth=1,
                   zeroline=True, zerolinecolor="rgba(203,213,225,0.8)", zerolinewidth=1,
                   tickfont=dict(size=13)),
        yaxis=dict(showgrid=False, zeroline=False, tickfont=dict(size=12)),
        hoverlabel=dict(bgcolor="white", bordercolor="rgba(100,116,139,0.3)",
                        font=dict(size=13, family=_FONT)),
    )

def _base_layout_hm(title, width=1200, height=700, a=0):
    base = _base_layout(title, width, height, lm=270, a=0)
    base["xaxis"] = dict(title_font=dict(size=15, color="#475569"), tickfont=dict(size=12),
                         showgrid=False, zeroline=False)
    base["yaxis"] = dict(title_font=dict(size=15, color="#475569"), tickfont=dict(size=11),
                         showgrid=False, zeroline=False)
    return base

def _dropdown(fig, buttons, x=0.99, y=1.11, lx=0.83, ly=1.11):
    fig.update_layout(updatemenus=[dict(
        type="dropdown", x=x, y=y, xanchor="right", yanchor="top", direction="down",
        buttons=buttons, bgcolor="#F1F5F9", bordercolor="rgba(148,163,184,0.6)",
        borderwidth=1, font=dict(size=13, family=_FONT, color="#1E293B"),
        pad=dict(r=8,t=4,b=4,l=8), showactive=True, active=0,
    )])
    fig.add_annotation(text="<b>A\u00f1o de an\u00e1lisis:</b>",
                       x=lx, y=ly, xref="paper", yref="paper",
                       xanchor="right", yanchor="top", showarrow=False,
                       font=dict(size=13, color="#475569", family=_FONT))

def _bm(color, opacity=0.93):
    """Bar marker with rounded corners and white separator line."""
    return dict(color=color, opacity=opacity, cornerradius=5,
                line=dict(color="rgba(255,255,255,0.65)", width=0.9))

def _smart_pad(max_val):
    """Proportional pad for total labels – avoids large gaps on short bars."""
    return max(float(max_val) * 0.012, 0.05)

def write_fig(fig, path):
    pio.write_html(fig, file=str(path), include_plotlyjs="cdn", full_html=True)

# ── 1) Overall papers per year ────────────────────────────────────────────────
overall_year_doc = (
    utb_planta_long[["EID","Year","doc_type3"]].drop_duplicates()
    .groupby(["Year","doc_type3"]).size().reset_index(name="n_docs")
)
overall_year_doc["doc_type3"] = pd.Categorical(
    overall_year_doc["doc_type3"], categories=DOC_TYPES_ORDER, ordered=True)
overall_year_doc = overall_year_doc.sort_values(["Year","doc_type3"])

fig_overall = go.Figure()
for dt, clr in DOC_TYPE_COLORS.items():
    sub = overall_year_doc[overall_year_doc["doc_type3"] == dt]
    fig_overall.add_trace(go.Bar(
        y=sub["Year"].tolist(), x=sub["n_docs"].tolist(), name=dt,
        orientation="h", marker=_bm(clr),
        hovertemplate="A\u00f1o=%{y}<br>Tipo="+dt+"<br>Docs=%{x}<extra></extra>"))

tot_year = overall_year_doc.groupby("Year")["n_docs"].sum().reset_index()
pad = _smart_pad(tot_year["n_docs"].max())
fig_overall.add_trace(go.Scatter(
    x=(tot_year["n_docs"]+pad).tolist(), y=tot_year["Year"].tolist(),
    mode="text", text=["<b>"+str(int(v))+"</b>" for v in tot_year["n_docs"]],
    textfont=dict(size=13, color="#1E293B"),
    showlegend=False, hoverinfo="skip", cliponaxis=False))
fig_overall.update_layout(
    **_base_layout("Documentos por a\u00f1o (\u2265 "+str(START_YEAR)+") \u2013 apilado por tipo", lm=90, a=0.2),
    barmode="stack")
fig_overall.update_xaxes(title_text="N\u00famero de documentos", title_font=dict(size=16,color="#475569"))
fig_overall.update_yaxes(title_text="A\u00f1o", title_font=dict(size=16,color="#475569"), type="category")
write_fig(fig_overall, OUT_DIR / "overall_papers_per_year_STACKED_by_type_horizontal.html")

# ── 2) Papers by Escuela ──────────────────────────────────────────────────────
fig_school = go.Figure()
trace_meta = []
school_orders = {}

for ys in year_sels:
    counts, order = school_doc_counts(ys)
    school_orders[ys] = order
    for dt in DOC_TYPES_ORDER:
        clr = DOC_TYPE_COLORS[dt]
        sub = counts[counts["doc_type3"].astype(str)==dt].copy()
        bdf = pd.DataFrame({"ESCUELA": order})
        sub = bdf.merge(sub[["ESCUELA","n_docs"]], on="ESCUELA", how="left").fillna({"n_docs":0})
        fig_school.add_trace(go.Bar(
            x=sub["n_docs"].tolist(), y=sub["ESCUELA"].tolist(), name=dt,
            orientation="h", visible=(ys=="ALL"), marker=_bm(clr),
            hovertemplate="Escuela=%{y}<br>Tipo="+dt+"<br>Docs=%{x}<extra></extra>"))
        trace_meta.append((ys, dt, len(fig_school.data)-1))
    tot = counts.groupby("ESCUELA")["n_docs"].sum().reindex(order, fill_value=0)
    pad = _smart_pad(tot.max())
    fig_school.add_trace(go.Scatter(
        x=(tot.values+pad).tolist(), y=order, mode="text",
        text=["<b>"+str(int(v))+"</b>" for v in tot.values],
        textfont=dict(size=12,color="#1E293B"), showlegend=False,
        hoverinfo="skip", visible=(ys=="ALL"), cliponaxis=False))
    trace_meta.append((ys, "__TOTAL__", len(fig_school.data)-1))

buttons = []
for ys in year_sels:
    vis = [False]*len(fig_school.data)
    for (y2,tag,idx) in trace_meta:
        if y2==ys: vis[idx]=True
    buttons.append(dict(label=str(ys), method="update", args=[
        {"visible": vis},
        {"title":{"text":"<b>Documentos por Escuela \u2013 apilado por tipo ("+ys+")</b>"},
         "yaxis":{"categoryorder":"array","categoryarray":school_orders[ys],"tickfont":{"size":12}},
         "xaxis":{"title":{"text":"N\u00famero de documentos"},
                  "showgrid":True,"gridcolor":"rgba(203,213,225,0.5)"}},
    ]))
fig_school.update_layout(**_base_layout("Documentos por Escuela \u2013 apilado por tipo", lm=280, a=0.1), barmode="stack")
fig_school.update_xaxes(title_text="N\u00famero de documentos", title_font=dict(size=16,color="#475569"))
fig_school.update_yaxes(title_text="Escuela", title_font=dict(size=16,color="#475569"),
                         categoryorder="array", categoryarray=school_orders["ALL"])
_dropdown(fig_school, buttons, y=1.25, ly=1.25)
write_fig(fig_school, OUT_DIR / "papers_by_escuela_STACKED_by_type_YEAR_dropdown.html")

# ── 3) Top authors ────────────────────────────────────────────────────────────
fig_auth = go.Figure()
trace_meta_a = []
author_orders = {}

for ys in year_sels:
    counts, order = author_doc_counts(ys)
    author_orders[ys] = order
    for dt in DOC_TYPES_ORDER:
        clr = DOC_TYPE_COLORS[dt]
        sub = counts[counts["doc_type3"].astype(str)==dt].copy()
        bdf = pd.DataFrame({"DOCENTE": order})
        sub = bdf.merge(sub[["DOCENTE","n_docs"]], on="DOCENTE", how="left").fillna({"n_docs":0})
        fig_auth.add_trace(go.Bar(
            x=sub["n_docs"].tolist(), y=sub["DOCENTE"].tolist(), name=dt,
            orientation="h", visible=(ys=="ALL"), marker=_bm(clr),
            hovertemplate="Autor=%{y}<br>Tipo="+dt+"<br>Docs=%{x}<extra></extra>"))
        trace_meta_a.append((ys, dt, len(fig_auth.data)-1))
    tot = counts.groupby("DOCENTE")["n_docs"].sum().reindex(order, fill_value=0)
    pad = _smart_pad(tot.max())
    fig_auth.add_trace(go.Scatter(
        x=(tot.values+pad).tolist(), y=order, mode="text",
        text=["<b>"+str(int(v))+"</b>" for v in tot.values],
        textfont=dict(size=12,color="#1E293B"), showlegend=False,
        hoverinfo="skip", visible=(ys=="ALL"), cliponaxis=False))
    trace_meta_a.append((ys, "__TOTAL__", len(fig_auth.data)-1))

buttons_a = []
for ys in year_sels:
    vis = [False]*len(fig_auth.data)
    for (y2,tag,idx) in trace_meta_a:
        if y2==ys: vis[idx]=True
    buttons_a.append(dict(label=str(ys), method="update", args=[
        {"visible": vis},
        {"title":{"text":"<b>Top "+str(TOP_AUTHORS)+" autores \u2013 apilado por tipo ("+ys+")</b>"},
         "yaxis":{"categoryorder":"array","categoryarray":author_orders[ys],"tickfont":{"size":12}},
         "xaxis":{"title":{"text":"N\u00famero de documentos"},
                  "showgrid":True,"gridcolor":"rgba(203,213,225,0.5)"}},
    ]))
fig_auth.update_layout(**_base_layout("Top "+str(TOP_AUTHORS)+" autores \u2013 apilado por tipo", lm=270), barmode="stack")
fig_auth.update_xaxes(title_text="N\u00famero de documentos", title_font=dict(size=16,color="#475569"))
fig_auth.update_yaxes(title_text="Autor", title_font=dict(size=16,color="#475569"),
                       categoryorder="array", categoryarray=author_orders["ALL"])
_dropdown(fig_auth, buttons_a)
write_fig(fig_auth, OUT_DIR / "top_authors_STACKED_by_type_YEAR_dropdown.html")

# ── 4) Heatmap ────────────────────────────────────────────────────────────────
_HM_CS = [[0.0,"#EFF6FF"],[0.25,"#BFDBFE"],[0.5,"#60A5FA"],[0.75,"#2563EB"],[1.0,"#1E3A8A"]]

fig_heat = go.Figure()
heat_orders = {}
for ys in year_sels:
    piv = heatmap_school_doctype(ys)
    heat_orders[ys] = list(piv.index)
    z = piv.values.tolist()
    txt = [[str(int(v)) if v>0 else "" for v in row] for row in z]
    fig_heat.add_trace(go.Heatmap(
        z=z, x=list(piv.columns), y=list(piv.index), text=txt,
        texttemplate="%{text}", textfont=dict(size=15, family=_FONT),
        visible=(ys=="ALL"), colorscale=_HM_CS, zmin=0,
        colorbar=dict(title=dict(text="Docs", font=dict(size=12,color="#475569"), side="right"),
                      thickness=16, len=0.72, yanchor="middle", y=0.5,
                      tickfont=dict(size=11), outlinewidth=0),
        hovertemplate="Escuela=%{y}<br>Tipo=%{x}<br>Docs=%{z}<extra></extra>"))

buttons_h = []
for i, ys in enumerate(year_sels):
    vis = [False]*len(fig_heat.data); vis[i]=True
    buttons_h.append(dict(label=str(ys), method="update", args=[
        {"visible": vis},
        {"title":{"text":"<b>Heatmap Escuela \u00d7 Tipo (+ totales) \u2013 "+ys+"</b>"},
         "yaxis":{"categoryorder":"array","categoryarray":heat_orders[ys],"tickfont":{"size":11}}},
    ]))
fig_heat.update_layout(**_base_layout_hm("Heatmap Escuela \u00d7 Tipo (+ totales)", height=680))
fig_heat.update_xaxes(title_text="Tipo de documento", title_font=dict(size=15,color="#475569"), tickfont=dict(size=13))
fig_heat.update_yaxes(title_text="Escuela", title_font=dict(size=15,color="#475569"), tickfont=dict(size=11),
                       categoryorder="array", categoryarray=heat_orders["ALL"])
_dropdown(fig_heat, buttons_h, y=1.19, ly=1.19)
write_fig(fig_heat, OUT_DIR / "heatmap_escuela_doctype_YEAR_dropdown.html")

# ── 5) Top pairs ──────────────────────────────────────────────────────────────
fig_pairs = go.Figure()
pair_orders = {}
for ys in year_sels:
    tp = top_pairs(ys)
    yv = tp["pair"].iloc[::-1].tolist() if not tp.empty else []
    xv = tp["n_shared"].iloc[::-1].tolist() if not tp.empty else []
    pair_orders[ys] = yv
    fig_pairs.add_trace(go.Bar(
        x=xv, y=yv, orientation="h", visible=(ys=="ALL"),
        marker=dict(color=xv, colorscale=[[0,"#BFDBFE"],[1,"#1D4ED8"]],
                    cornerradius=5, line=dict(color="rgba(255,255,255,0.5)",width=0.8), showscale=False),
        hovertemplate="Par=%{y}<br>Docs compartidos=%{x}<extra></extra>", showlegend=False))

buttons_p = []
for i, ys in enumerate(year_sels):
    vis = [False]*len(fig_pairs.data); vis[i]=True
    buttons_p.append(dict(label=str(ys), method="update", args=[
        {"visible": vis},
        {"title":{"text":"<b>Top "+str(TOP_PAIRS)+" pares (planta\u2013planta) \u2013 "+ys+"</b>"},
         "yaxis":{"categoryorder":"array","categoryarray":pair_orders[ys],"tickfont":{"size":11}}},
    ]))
fig_pairs.update_layout(**_base_layout("Top "+str(TOP_PAIRS)+" pares (planta\u2013planta)", lm=390))
fig_pairs.update_xaxes(title_text="Documentos compartidos", title_font=dict(size=16,color="#475569"))
fig_pairs.update_yaxes(title_text="Par de docentes", title_font=dict(size=16,color="#475569"),
                        categoryorder="array", categoryarray=pair_orders["ALL"])
_dropdown(fig_pairs, buttons_p)
write_fig(fig_pairs, OUT_DIR / "top_pairs_faculty_faculty_YEAR_dropdown.html")

# ── All-in-one index.html ─────────────────────────────────────────────────────
_ECFG = {"responsive": True, "displayModeBar": True,
         "modeBarButtonsToRemove": ["select2d","lasso2d"], "displaylogo": False}

def _embed(fig, height, lm=None):
    d = fig.to_dict()
    d["layout"]["height"] = height
    d["layout"].pop("width", None)
    d["layout"]["autosize"] = True
    mg = d["layout"].get("margin", {})
    if isinstance(mg, dict):
        if lm is not None: mg["l"] = lm
        mg["r"] = 12
        d["layout"]["margin"] = mg
    return pio.to_html(go.Figure(d), include_plotlyjs=False,
                       full_html=False, config=_ECFG)

_dov = _embed(fig_overall, height=360, lm=80)
_dhm = _embed(fig_heat,    height=440, lm=175)
_dsc = _embed(fig_school,  height=400, lm=210)
_dau = _embed(fig_auth,    height=620, lm=210)
_dpa = _embed(fig_pairs,   height=700, lm=340)

_ndocs  = int(utb_planta_long[["EID"]].drop_duplicates().shape[0])
_nauth  = int(utb_planta_long[["author_id"]].drop_duplicates().shape[0])
_nsch   = int(utb_planta_long["ESCUELA"].nunique())
_yr     = str(int(utb_planta_long["Year"].min())) + "\u2013" + str(int(utb_planta_long["Year"].max()))

_INDEX_TEMPLATE = '<!doctype html>\n<html lang="es">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <title>UTB Scopus Dashboard >= __START_YEAR__</title>\n  <script src="https://cdn.plot.ly/plotly-3.3.1.min.js" crossorigin="anonymous"></script>\n  <style>\n    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }\n    html { scroll-behavior: smooth; }\n    body {\n      font-family: \'Segoe UI\', system-ui, -apple-system, Arial, sans-serif;\n      background: #EEF2F8;\n      color: #1E293B;\n      -webkit-font-smoothing: antialiased;\n    }\n\n    \
/* ── HERO ─────────────────────────────────────────────────────────── */\n    .hero {\n      background: linear-gradient(135deg, #0A1628 0%, #0F2D6B 45%, #1D4ED8 100%);\n      padding: 60px 80px 80px;\n      position: relative;\n      overflow: hidden;\n      color: white;\n    }\n    .hero::before {\n      content: \'\';\n      position: absolute; inset: 0;\n      background-image: url("data:image/svg+xml,%3Csvg width=\'40\' height=\'40\' viewBox=\'0 0 40 40\' xmlns=\'http://www.w3.org/2000/svg\'%3E%3Cg fill=\'%23ffffff\' fill-opacity=\'0.03\'%3E%3Ccircle cx=\'20\' cy=\'20\' r=\'1.5\'/%3E%3C/g%3E%3C/svg%3E");\n    }\n    .hero-glow-1 {\n      position: absolute; top: -150px; right: -50px;\n      width: 550px; height: 550px; border-radius: 50%;\n      background: radial-gradient(circle, rgba(99,102,241,0.30) 0%, transparent 65%);\n      pointer-events: none;\n    }\n    .hero-glow-2 {\n      position: absolute; bottom: -100px; left: 20%;\n      width: 350px; height: 350px; border-radius: 50%;\n      background: radial-gradient(circle, rgba(14,165,233,0.20) 0%, transparent 65%);\n      pointer-events: none;\n    }\n    .hero-inner { position: relative; z-index: 2; max-width: 1240px; margin: 0 auto; }\n    .hero-badge {\n      display: inline-flex; align-items: center; gap: 7px;\n      background: rgba(255,255,255,0.11); border: 1px solid rgba(255,255,255,0.22);\n      border-radius: 999px; padding: 5px 15px; font-size: 12px; font-weight: 600;\n      letter-spacing: 0.3px; margin-bottom: 20px; backdrop-filter: blur(8px);\n    }\n    .hero-title {\n      font-size: 40px; font-weight: 800; letter-spacing: -1.2px; line-height: 1.1;\n      margin-bottom: 16px;\n    }\n    .hero-title .accent { color: #93C5FD; }\n    .hero-desc {\n      font-size: 14.5px; line-height: 1.75; color: rgba(255,255,255,0.76);\n      max-width: 740px;\n    }\n    .hero-desc strong { color: rgba(255,255,255,0.94); }\n\n    \
/* ── KPI STRIP ────────────────────────────────────────────────────── */\n    .kpi-strip {\n      max-width: 1240px; margin: -34px auto 0;\n      padding: 0 80px; position: relative; z-index: 10;\n    }\n    .kpi-grid { display: grid; grid-template-columns: repeat(4,1fr); gap: 16px; }\n    .kpi-card {\n      background: #fff; border-radius: 14px; padding: 22px 24px 17px;\n      box-shadow: 0 8px 32px rgba(15,23,42,0.11);\n      border-top: 20px solid var(--accent);\n      transition: transform 0.18s ease, box-shadow 0.18s ease;\n      animation: riseIn 0.55s ease both;\n    }\n    .kpi-card:hover { transform: translateY(-4px); box-shadow: 0 16px 48px rgba(15,23,42,0.16); }\n    .kpi-card:nth-child(1) { --accent:#2563EB; animation-delay:0.06s; }\n    .kpi-card:nth-child(2) { --accent:#059669; animation-delay:0.12s; }\n    .kpi-card:nth-child(3) { --accent:#EA580C; animation-delay:0.18s; }\n    .kpi-card:nth-child(4) { --accent:#7C3AED; animation-delay:0.24s; }\n    .kpi-label {\n      font-size: 10.5px; font-weight: 700; text-transform: uppercase;\n      letter-spacing: 0.9px; color: #94A3B8; margin-bottom: 7px;\n    }\n    .kpi-value { font-size: 36px; font-weight: 800; line-height: 1; color: #0F172A; }\n    .kpi-value.md { font-size: 27px; }\n    .kpi-sub { font-size: 11.5px; color: #94A3B8; margin-top: 5px; }\n\n    \
/* ── SECTION ──────────────────────────────────────────────────────── */\n    .section { padding: 52px 80px 72px; max-width: 1240px; margin: 0 auto; }\n    .section-eyebrow {\n      font-size: 11px; font-weight: 700; text-transform: uppercase;\n      letter-spacing: 1.1px; color: #94A3B8; margin-bottom: 10px;\n    }\n    .section-heading {\n      font-size: 22px; font-weight: 700; color: #0F172A;\n      margin-bottom: 10px; letter-spacing: -0.4px;\n    }\n    .section-note {\n      background: #FEF9EC; border: 1px solid #FDE68A; border-left: 4px solid #F59E0B;\n      border-radius: 8px; padding: 13px 18px; font-size: 13px;\n      line-height: 1.65; color: #78350F; margin-bottom: 32px;\n    }\n\n    \
/* ── CHART GRID ───────────────────────────────────────────────────── */\n    .chart-grid {\n      display: grid;\n      grid-template-columns: 1fr;\n      gap: 26px;\n    }\n    .chart-card {\n      background: #fff; border-radius: 16px;\n      box-shadow: 0 4px 24px rgba(15,23,42,0.08);\n      overflow: hidden;\n      transition: box-shadow 0.2s ease, transform 0.2s ease;\n      animation: riseIn 0.55s ease both;\n      border: 1px solid rgba(226,232,240,0.7);\n    }\n    .chart-card:hover { box-shadow: 0 14px 52px rgba(15,23,42,0.14); transform: translateY(-2px); }\n    .chart-card:nth-child(1) { animation-delay: 0.05s; }\n    .chart-card:nth-child(2) { animation-delay: 0.10s; }\n    .chart-card:nth-child(3) { animation-delay: 0.15s; }\n    .chart-card:nth-child(4) { animation-delay: 0.20s; }\n    .chart-card:nth-child(5) { animation-delay: 0.25s; }\n    .chart-card.full { grid-column: 1 / -1; }\n    .chart-header {\n      display: flex; align-items: center; gap: 10px;\n      padding: 16px 22px 4px;\n    }\n    .chart-dot { width: 8px; height: 8px; border-radius: 50%; flex-shrink: 0; }\n    .chart-label {\n      font-size: 11.5px; font-weight: 700; text-transform: uppercase;\n      letter-spacing: 0.7px; color: #64748B;\n    }\n    /* Force Plotly to fill the card width */\n    .chart-body .plotly-graph-div { width: 100% !important; }\n    .chart-body { line-height: 0; }\n\n    \
/* ── FOOTER ───────────────────────────────────────────────────────── */\n    .site-footer {\n      background: #0F172A; color: rgba(255,255,255,0.52);\n      padding: 38px 80px; font-size: 13px; line-height: 1.75;\n    }\n    .site-footer strong { color: rgba(255,255,255,0.82); }\n\n    /* ── ANIMATIONS ───────────────────────────────────────────────────── */\n    @keyframes riseIn {\n      from { opacity: 0; transform: translateY(22px); }\n      to   { opacity: 1; transform: translateY(0); }\n    }\n\n    \
/* ── RESPONSIVE ───────────────────────────────────────────────────── */\n    @media (max-width: 1100px) {\n      .hero { padding: 44px 36px 74px; }\n      .kpi-strip { padding: 0 36px; }\n      .kpi-grid  { grid-template-columns: repeat(2,1fr); }\n      .section   { padding: 44px 36px; }\n      .chart-grid { grid-template-columns: 1fr; }\n    }\n    @media (max-width: 640px) {\n      .hero-title { font-size: 28px; }\n      .kpi-grid { grid-template-columns: 1fr 1fr; }\n    }\n  </style>\n</head>\n<body>\n\n<!-- HERO -->\n<header class="hero">\n  <div class="hero-glow-1"></div>\n  <div class="hero-glow-2"></div>\n  <div class="hero-inner">\n    <div class="hero-badge">&#128202; Actualizado __ACTU__</div>\n    <h1 class="hero-title">UTB Scopus Dashboard <span class="accent">&#8805; __START_YEAR__</span></h1>\n    <p class="hero-desc">\n      Caracterizaci&#243;n bibliom&#233;trica de la producci&#243;n cient&#237;fica asociada a docentes de planta\n      de la <strong>Universidad Tecnol&#243;gica de Bol&#237;var</strong> a partir de datos de Scopus.\n      Conteos sobre documentos &#250;nicos (EID) con desagregaci&#243;n por escuela, tipo y a&#241;o.\n    </p>\n  </div>\n</header>\n\n<!-- KPI STRIP -->\n<div class="kpi-strip">\n  <div class="kpi-grid">\n    <div class="kpi-card">\n      <div class="kpi-label">Documentos &#250;nicos</div>\n      <div class="kpi-value">__N_DOCS__</div>\n      <div class="kpi-sub">desde __START_YEAR__</div>\n    </div>\n    <div class="kpi-card">\n      <div class="kpi-label">Docentes activos</div>\n      <div class="kpi-value">__N_AUTHORS__</div>\n      <div class="kpi-sub">con publicaciones</div>\n    </div>\n    <div class="kpi-card">\n      <div class="kpi-label">Escuelas</div>\n      <div class="kpi-value">__N_SCHOOLS__</div>\n      <div class="kpi-sub">con producci&#243;n</div>\n    </div>\n    <div class="kpi-card">\n      <div class="kpi-label">Per&#237;odo</div>\n      <div class="kpi-value md">__YR_RANGE__</div>\n      <div class="kpi-sub">a&#241;os analizados</div>\n    </div>\n  </div>\n</div>\n\n<!-- CHARTS -->\n<main class="section">\n  <p class="section-eyebrow">&#128202; Visualizaciones interactivas</p>\n <div class="repo-description" style="margin-bottom: 24px; line-height: 1.6; color: #475569; font-size: 12px;"><p>Este repositorio contiene un tablero interactivo (HTML) y tablas (Excel) para explorar la producción científica asociada a docentes de planta de la Universidad Tecnológica de Bolívar (UTB) a partir de un export de Scopus.</p> </br> <p style="background: #F1F5F9; padding: 12px 16px; border-left: 10px solid #2563EB; border-radius: 8px;"><strong>Actualización:</strong> estas estadísticas y gráficos están actualizados a fecha de <strong>__ACTU__</strong>.</p> </br> \n <p style="background: #D0F5E9; padding: 12px 16px; border-left: 10px solid #059669; border-radius: 8px;"> <strong>Metodología (Resumen)...</strong> <strong>Fuente de datos:</strong> export CSV desde Scopus (EID, año, tipo de documento, autores). <strong>Vinculación a planta UTB:</strong> cruce por <strong>Scopus Author ID</strong> contra una base maestra interna de docentes de planta. <strong>Unidad de conteo:</strong> documentos únicos por <strong>EID</strong> (evita dobles conteos por múltiples apareciones del mismo autor). <strong>Crédito por Escuela:</strong> una Escuela recibe crédito si al menos un docente de esa Escuela aparece como autor en el documento (un documento puede contar en más de una Escuela si hay coautoría inter-escuelas). <strong>Tipos de documento:</strong> agrupación operativa en <em>Article</em>, <em>Conference</em>, <em>Review</em> y <em>Other</em> según “Document Type”.</p></div> \n  <h2 class="section-heading">Producci&#243;n cient&#237;fica UTB</h2>\n  <div class="section-note">\n    &#128161; Usa el men&#250; <b>"A&#241;o de an&#225;lisis"</b> en cada gr&#225;fica para filtrar por a&#241;o.\n    Haz clic en la leyenda para mostrar/ocultar tipos. Hover sobre las barras para ver detalles.\n  </div>\n\n  <div class="chart-grid">\n\n    <!-- 1: Produccion anual -->\n    <div class="chart-card">\n      <div class="chart-header">\n        <span class="chart-dot" style="background:#2563EB"></span>\n        <span class="chart-label">Producci&#243;n anual total</span>\n      </div>\n      <div class="chart-body">CHART_OVERALL_PLACEHOLDER</div>\n    </div>\n\n    <!-- 2: Heatmap -->\n    <div class="chart-card">\n      <div class="chart-header">\n        <span class="chart-dot" style="background:#7C3AED"></span>\n        <span class="chart-label">Heatmap escuela &#215; tipo</span>\n      </div>\n      <div class="chart-body">CHART_HEATMAP_PLACEHOLDER</div>\n    </div>\n\n    <!-- 3: Escuelas -->\n    <div class="chart-card">\n      <div class="chart-header">\n        <span class="chart-dot" style="background:#059669"></span>\n        <span class="chart-label">Producci&#243;n por escuela</span>\n      </div>\n      <div class="chart-body">CHART_SCHOOL_PLACEHOLDER</div>\n    </div>\n\n    <!-- 4: Autores -->\n    <div class="chart-card">\n      <div class="chart-header">\n        <span class="chart-dot" style="background:#EA580C"></span>\n        <span class="chart-label">Top autores</span>\n      </div>\n      <div class="chart-body">CHART_AUTHORS_PLACEHOLDER</div>\n    </div>\n\n    <!-- 5: Pares (full width) -->\n    <div class="chart-card full">\n      <div class="chart-header">\n        <span class="chart-dot" style="background:#0891B2"></span>\n        <span class="chart-label">Pares de coautor&#237;a (planta&#8211;planta)</span>\n      </div>\n      <div class="chart-body">CHART_PAIRS_PLACEHOLDER</div>\n    </div>\n\n  </div>\n</main>\n\n<!-- FOOTER -->\n<footer class="site-footer">\n  <strong>Metodolog&#237;a:</strong> Fuente CSV desde Scopus. Vinculaci&#243;n a planta UTB mediante\n  <em>Scopus Author ID</em>. Conteos sobre documentos &#250;nicos (EID). La desagregaci&#243;n por Escuela\n  asigna cr&#233;dito cuando al menos un docente de esa Escuela participa como autor.\n  <strong>Actualizado:</strong> __ACTU__ &nbsp;&#183;&nbsp; <strong>Per&#237;odo:</strong> desde __START_YEAR__. <strong>Nota:</strong>Este tablero es un ejercicio técnico y personal de análisis bibliométrico basado en un export puntual de Scopus y una lista interna de docentes de planta. Los resultados son referenciales y pueden diferir de cifras institucionales oficiales (cobertura del export, actualización de perfiles, homónimos/duplicados de Author ID, reglas de conteo). Este material no constituye un reporte oficial ni representa una posición institucional de la UTB. <strong>Desarrollado por:</strong> D. Sierra-Porta © 2026 — Universidad Tecnológica de Bolívar\n</footer>\n\n</body>\n</html>'


_html = (_INDEX_TEMPLATE
    .replace("CHART_OVERALL_PLACEHOLDER", _dov)
    .replace("CHART_HEATMAP_PLACEHOLDER", _dhm)
    .replace("CHART_SCHOOL_PLACEHOLDER",  _dsc)
    .replace("CHART_AUTHORS_PLACEHOLDER", _dau)
    .replace("CHART_PAIRS_PLACEHOLDER",   _dpa)
    .replace("__N_DOCS__",    str(_ndocs))
    .replace("__N_AUTHORS__", str(_nauth))
    .replace("__N_SCHOOLS__", str(_nsch))
    .replace("__YR_RANGE__",  _yr)
    .replace("__START_YEAR__", str(START_YEAR))
    .replace("__ACTU__",      str(Actu))
)
(OUT_DIR / "index.html").write_text(_html, encoding="utf-8")
print("index.html written,", len(_html)//1024, "KB")


index.html written, 151 KB


In [8]:
# ---- tables + index ----
tables_path = OUT_DIR / "tables.xlsx"
with pd.ExcelWriter(tables_path, engine="openpyxl") as w:
    overall_year_doc.rename(columns={"n_docs":"n_documentos"}).to_excel(w, sheet_name="overall_year_docType", index=False)
    planta_school_papers.to_excel(w, sheet_name="planta_school_papers", index=False)
    planta_author_papers.to_excel(w, sheet_name="planta_author_papers", index=False)
    pairs_counts_year.to_excel(w, sheet_name="pairs_counts_year", index=False)
    pairs_counts_all.to_excel(w, sheet_name="pairs_counts_all", index=False)
    faculty_missing_id.to_excel(w, sheet_name="faculty_missing_id", index=False)

In [9]:
# index.html now generated inside the visualization cell above
pass

In [10]:
print("DONE ✅")
print(f"Open: {OUT_DIR / 'index.html'}")

DONE ✅
Open: utb_scopus_dashboard_single_pretty/index.html


In [11]:
from pathlib import Path

FOLDER = "utb_scopus_dashboard_single_pretty"

In [12]:
readme_md = f"""# UTB Scopus Dashboard (>= __START_YEAR__)

Este repositorio contiene un tablero interactivo (HTML) y tablas (Excel) para explorar la producción científica asociada a docentes de planta de la
Universidad Tecnológica de Bolívar (UTB) a partir de un export de Scopus.

> **Actualización:** estas estadísticas y gráficos están actualizados a fecha de **__ACTU__**.

## Metodología (resumen)

- **Fuente de datos:** export CSV desde Scopus (EID, año, tipo de documento, autores).
- **Vinculación a planta UTB:** cruce por **Scopus Author ID** contra una base maestra interna de docentes de planta.
- **Unidad de conteo:** documentos únicos por **EID** (evita dobles conteos por múltiples apariciones del mismo autor).
- **Crédito por Escuela:** una Escuela recibe crédito si al menos un docente de esa Escuela aparece como autor en el documento (un documento puede contar en más de una Escuela si hay coautoría inter-escuelas).
- **Tipos de documento:** agrupación operativa en *Article*, *Conference*, *Review* y *Other* según “Document Type”.

## Abrir el tablero

Abre el archivo principal:

- **Página principal (índice):** `{FOLDER}/index.html`

## Gráficos y estadísticas

- **Documentos por año (apilado por tipo):** `{FOLDER}/overall_papers_per_year_STACKED_by_type_horizontal.html`
- **Documentos por Escuela (apilado + dropdown por año):** `{FOLDER}/papers_by_escuela_STACKED_by_type_YEAR_dropdown.html`
- **Top autores (apilado + dropdown por año):** `{FOLDER}/top_authors_STACKED_by_type_YEAR_dropdown.html`
- **Heatmap Escuela × Tipo (con totales + dropdown por año):** `{FOLDER}/heatmap_escuela_doctype_YEAR_dropdown.html`
- **Top pares planta–planta (dropdown por año):** `{FOLDER}/top_pairs_faculty_faculty_YEAR_dropdown.html`

## Tablas (Excel)

- **Descargar tablas:** `{FOLDER}/tables.xlsx`

## Aclaratoria

Este tablero es un ejercicio técnico y personal de análisis bibliométrico basado en un export puntual de Scopus y una lista interna de docentes de planta.
Los resultados son **referenciales** y pueden diferir de cifras institucionales oficiales (cobertura del export, actualización de perfiles, homónimos/duplicados de Author ID, reglas de conteo).
Este material no constituye un reporte oficial ni representa una posición institucional de la UTB.

## Créditos
Desarrollado por **D. Sierra-Porta** © 2026 — Universidad Tecnológica de Bolívar
"""



In [13]:
# Reemplazos
readme_md = readme_md.replace("__START_YEAR__", str(START_YEAR)).replace("__ACTU__", str(Actu))

# Guardar en la raíz del repo
Path("README.md").write_text(readme_md, encoding="utf-8")
print("Wrote README.md")

readme_md = readme_md.replace("__START_YEAR__", str(START_YEAR)).replace("__ACTU__", str(Actu))
Path("README.md").write_text(readme_md, encoding="utf-8")

Wrote README.md


2540